## PlantTraits



### Project

This end of course project present you with a real-life application of Machine Learning on plant science data. 

The goal of this project is to predict plant trait characteristics from climate and soil features, satellite features and photographs. The task and data was first published for the CVPR 2024 FGVC11 (Fine-Grained Visual Categorization) Workshop and is hosted on Kaggle. 

This notebook guides you through a few steps and different design to solve the task. The project is kept quite open-ended and meant for you to explore solutions. Feel free to spend more time and effort on the portion that are most interesting to you. We do not expect nor aim for your solutions to compete with the Kaggle competition benchmark (optimizing hyperparameters is not within the scope of the lab). 

The notebook includes several **questions** text cell. Please fill those in your own words. The question are made to assess your understanding. They are also open-ended and do not have single right answer. 

If you have data and a project you'd prefer to work on, you are encouraged to do so. The only requirement is for you to work on novel Machine Learning methods presented during the course. Make sure to present your project to us so that we can make sure the project is a fit and advise you on the scope and direction to take. 

For submission of this final project required to validate the credits for the course, please share with the TAs your work before December 4th (extensions may be awarded if justified). Prefered format is a private github repository shared with the TAs (TGazel). Zip files via emails are also accepted. 

### Data source

This project uses the data of the [PlantTraits2024 Kaggle competition](https://www.kaggle.com/competitions/planttraits2024), which combines:
- plant photographs shared by citizen scientists on [iNaturalist](https://www.inaturalist.org/)
- plant trait values from the [TRY plant trait database](https://www.try-db.org/)
- environmental data read from global maps at the location of each photograph

Predicting plant traits by combining iNaturalist photographs with TRY data is described in this paper:

> Schiller, C., Schmidtlein, S., Boonman, C., Moreno-Martínez, A., Kattenborn, T. (2021). [Deep learning and citizen science enable automated plant trait predictions from photographs](https://www.nature.com/articles/s41598-021-95616-0). *Scientific Reports*, 11, 16395.

### Task

The task for this project is to **predict six plant traits from a photograph of a plant and from environmental data at the location where the photograph was taken**.

## Dataset

The dataset contains 55,489 samples, one per photograph.

**Targets:** the six traits to predict.

| Target column | Trait | Unit |
|---|---|---|
| `X4_mean` | Stem specific density (wood density) | g/cm³ |
| `X11_mean` | Specific leaf area (leaf area per leaf dry mass) | mm²/mg |
| `X18_mean` | Plant height | m |
| `X26_mean` | Seed dry mass | mg |
| `X50_mean` | Leaf nitrogen content per leaf area | g/m² |
| `X3112_mean` | Leaf area | mm² |

**Inputs:** a photograph of 512x512 pixels, and 163 environmental variables.

| Input columns | Number | Content |
|---|---|---|
| `WORLDCLIM_*` | 6 | Climate: temperature and precipitation |
| `SOIL_*` | 61 | Soil properties (for example sand, clay, pH, nitrogen) at different depths |
| `MODIS_*` | 60 | Optical satellite data: surface reflectance, 5 spectral bands x 12 months |
| `VOD_*` | 36 | Radar satellite data: vegetation optical depth, 3 frequency bands x 12 months |

The table also contains the columns `X4_sd`, ..., `X3112_sd`: the standard deviation of each trait. They describe the targets and would not be known for a new photograph, so they cannot be used as model inputs.

## Data download

The dataset is hosted on Kaggle. The download is 3.2 GB, and you need about 7 GB of free disk space to download and unzip it.

**1. Accept the competition rules.** Create a free [Kaggle](https://www.kaggle.com/) account if you do not have one, then open the [competition rules](https://www.kaggle.com/competitions/planttraits2024/rules) and accept them. Without this step, the download is refused.

**2. Download the data**, with one of these two options:

- **Option A, zip download:** on the [Data tab](https://www.kaggle.com/competitions/planttraits2024/data) of the competition, download all the files as `planttraits2024.zip`, and move this file to the folder `5-PlantTraits/data/planttraits2024`.
- **Option B, API key:** in your [Kaggle settings](https://www.kaggle.com/settings), create a new API token and copy it. Keep this token private. Then, in a terminal with the course environment activated, run from the folder `5-PlantTraits`:

```bash
pip install kaggle six
export KAGGLE_API_TOKEN=your_token    # on Windows (Anaconda Prompt): set KAGGLE_API_TOKEN=your_token
kaggle competitions download -c planttraits2024 -p data/planttraits2024
```

**3. Unzip** `planttraits2024.zip` in the folder `5-PlantTraits/data/planttraits2024`, to obtain the structure below. You can then delete the zip file, as well as the files `test.csv`, `sample_submission.csv` and the folder `test_images/`: the test set of the competition has no public target values, so we do not use it.

```
5-PlantTraits/
├── 5-PlantTraits.ipynb
├── utils.py
└── data/
    └── planttraits2024/
        ├── train.csv
        ├── target_name_meta.tsv
        └── train_images/
```

**Note:** `train.csv` and `train_images/` are the names given by Kaggle. In this project, they contain the **whole dataset**, which you will split yourself.

## Data preparation

Functions to load and display data are in the file `utils.py`, next to this notebook.

In [ ]:
import copy
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader
from torchvision import models, transforms

from utils import PlantDataset, load_data, load_embeddings, load_image, show_sample, show_samples

torch.manual_seed(42)  # for reproducibility
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
data_df, target_cols, feature_cols = load_data()

image = load_image(data_df["id"].iloc[0])
print(f"Image size: {image.size}")

In [ ]:
# Some images and their target values
show_samples(data_df, random_state=42)

# One image, with its target values and climate data
show_sample(data_df.iloc[0])

### Splitting the data

To evaluate the models on samples they have not seen, `data_df` is split between a training set, a validation set and a test set. The validation set is used to compare models and settings, for example to choose the best training epoch in section 3, and the test set only to evaluate the final models.

The positions (row numbers in `data_df`) of the samples of each set are stored in `train_idx`, `val_idx` and `test_idx`.

In [ ]:
# Photographs with identical target values belong to the same species
species = data_df.groupby(target_cols).ngroup().to_numpy()

# 20% of the species for testing, 10% for validation and 70% for training
outer_split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_val_idx, test_idx = next(outer_split.split(data_df, groups=species))
inner_split = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=42)
train_pos, val_pos = next(inner_split.split(train_val_idx, groups=species[train_val_idx]))
train_idx, val_idx = train_val_idx[train_pos], train_val_idx[val_pos]

for name, idx in [("Training", train_idx), ("Validation", val_idx), ("Test", test_idx)]:
    print(f"{name}: {len(idx)} samples, {len(set(species[idx]))} species")

In [ ]:
# TODO Explore target data and handle outliers and missing values

#### Questions
- Why is the data split this way, rather than with a random split of the samples?
- Which metrics can you use to assess performance for this task?
- Which data cleaning steps did you implement? Why?


## Models


### 1: climate and soil data

Start with a simple model using only the climate (`WORLDCLIM_*`) and soil (`SOIL_*`) columns.

In [ ]:
# TODO train a simple model on climate and soil data

#### Questions
- Explain your model choice. Explain how you prepared the input data and why.

### 2: adding satellite imagery data

Train a machine learning model on all the environmental variables, including the satellite data (`MODIS_*` and `VOD_*`). 

You may train several model types and configurations.

In [ ]:
# TODO train on all tabular data 

# TODO compare performance between models

#### Questions
- Explain your model choice. Explain how you prepared the input data and why.
- Is the tabular data useful? In your opinion which data is most important?
- Do you believe this model answers the original task adequately? Do you see any pitfalls to the design presented here? 

### 3:  images

Train data includes an image for each entry. Build a Computer Vision model using images only.

You may train several model types and configurations. If training takes too long on your personal laptop, consider lighter solutions such as fine-tuning only. You are allowed and encouraged to research and use state-of-the-art solutions, if you want to. Image embeddings were provided to you by email, if you prefer using these. 

**Given code:** the cells below implement the data preparation and the training procedure, so that you only need to define your model:
- `eval_transform` prepares the images for a model pretrained on ImageNet, and `train_transform` also changes them randomly for training (data augmentation).
- `train_loader`, `val_loader` and `test_loader` give batches of `(images, tabular, targets)` for the samples of `train_idx`, `val_idx` and `test_idx`, with your prepared targets `y`.
- `train_model(model, learning_rate)` trains a model on `train_loader`, and keeps the weights of the epoch with the best R² on `val_loader`.
- `predict(model, loader)` returns the predictions of a model for all the samples of a loader.

Your model must be a PyTorch `nn.Module` whose `forward(images, tabular)` method returns the six predicted targets of each sample. A model on images only simply ignores `tabular`, and the same code can be reused in section 4. You may adapt the transforms and the training settings (`EPOCHS`, `BATCH_SIZE`, learning rate). 

#### Questions
- Explain the training procedure below. How is the data prepared? How is the model trained?

In [ ]:
# Models pretrained on ImageNet expect images of 224x224 pixels, normalized with the ImageNet statistics
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# For evaluation: resize, then crop the center of the image
eval_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# For training
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print(f"Transformed image: {tuple(eval_transform(image).shape)}")

In [ ]:
EPOCHS = 12
BATCH_SIZE = 128
NUM_WORKERS = 4  # processes loading the images in parallel (set to 0 if this causes errors)

# Input columns for the neural networks, standardized with the training set and clipped
X_tabular = data_df[feature_cols].to_numpy()
tabular_scaler = StandardScaler().fit(X_tabular[train_idx])
X_tabular_scaled = np.clip(tabular_scaler.transform(X_tabular), -5, 5).astype(np.float32)


def make_dataset(indices, transform):
    """Photographs, tabular inputs and targets of the samples at the given positions in data_df (PlantDataset is in utils.py)."""
    return PlantDataset(data_df["id"].to_numpy()[indices], X_tabular_scaled[indices], y[indices], transform)


train_loader = DataLoader(make_dataset(train_idx, train_transform), batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=device == "cuda", persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(make_dataset(val_idx, eval_transform), batch_size=256, num_workers=NUM_WORKERS, pin_memory=device == "cuda")
test_loader = DataLoader(make_dataset(test_idx, eval_transform), batch_size=256, num_workers=NUM_WORKERS, pin_memory=device == "cuda")

In [ ]:
@torch.no_grad()
def predict(model, loader):
    """Predict the targets of all the samples of a loader."""
    model.eval()
    outputs = []
    for images, tabular, _ in loader:
        with torch.amp.autocast(device_type=device, enabled=device == "cuda"):
            outputs.append(model(images.to(device), tabular.to(device)).float().cpu())
    return torch.cat(outputs).numpy()


def train_model(model, learning_rate=1e-4):
    """Train a model, and keep the weights of its best epoch on the validation set."""
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler(device, enabled=device == "cuda")

    best_score, best_weights = -np.inf, None
    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.perf_counter()
        model.train()
        losses = []
        for images, tabular, targets in train_loader:
            images, tabular, targets = images.to(device), tabular.to(device), targets.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type=device, enabled=device == "cuda"):
                loss = nn.functional.mse_loss(model(images, tabular), targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())
        scheduler.step()

        score = r2_score(y[val_idx], predict(model, val_loader))
        if score > best_score:
            best_score, best_weights = score, copy.deepcopy(model.state_dict())
        print(f"Epoch {epoch:2d}: training loss {np.mean(losses):.3f}, validation R² {score:.3f}, {time.perf_counter() - epoch_start:.0f} s")

    model.load_state_dict(best_weights)
    return model

In [ ]:
class ImageRegressor(nn.Module):
    """Model predicting the six targets from the images."""

    def __init__(self):
        super().__init__()
        # TODO define the layers of the model, for example a model pretrained on ImageNet
        # from torchvision.models, with its last layer replaced by a layer predicting the six targets

    def forward(self, images, tabular):
        # TODO return the predicted targets, a tensor of shape (number of images, 6).
        # Do not use tabular data in this model, only the images.
        pass


image_model = train_model(ImageRegressor(), learning_rate=1e-4)
test_pred = predict(image_model, test_loader)

# TODO evaluate test_pred and compare performance with the previous results

#### Foundation models and embeddings

State-of-the-art foundation computer vision models are very good at extracting all important features from an image. 

DINOv2 embeddings for the images are available (shared via email) and may be used instead of the images themselves. Unzip the embedding data in the folder `5-PlantTraits/data/`, then load the embeddings with `load_embeddings(data_df)` from `utils.py`. This gives 1,024 values per image, in the same order as the rows of `data_df`.

DINOv2 is described in this paper:

> Oquab, M., Darcet, T., Moutakanni, T., et al. (2023). [DINOv2: Learning Robust Visual Features without Supervision](https://arxiv.org/abs/2304.07193). *arXiv:2304.07193*.

In [ ]:
# TODO define a Computer Vision model on images using the embeddings

# TODO compare performance with the previous results

#### Questions
- Are images useful for prediction? Are they enough by themself for prediction?
- If you used the DINOv2 embeddings, are they useful here? How do they compare with your other image models?


### 4: using all modalites

Build a model that uses all available data. The model should take as input both the imagery and the tabular data. 

You may train several model types and configurations. Look into simple solutions first, e.g. combining outputs of earlier models. If training takes too long on your personal laptop, consider lighter solutions such as fine-tuning only orc using embeddings. You are allowed and encouraged to research and use state-of-the-art solutions, if you want to.

In [ ]:
# TODO design a solution that combines tabular and image data

# TODO compare performance with the previous results

#### Questions
- Explain your model choice and training procedure. How do you prepare the data? How do you train (learning rate, augmentations...)? Do you use a pretrained model?
-  Is all the data useful? In your opinion which data is most important, and which is not needed?

### 5: (optional) submit your result to the official test set on Kaggle!

The Kaggle competition has a private test dataset for fair comparison of methods. You can submit your predictions on the test set and see how well they perform!

### Conclusion
#### Questions
- In this scenario, which solution works better? Can you explain why (model, data...)?
- Are there any issues with the dataset? If so, which ones? Can you think of how to improve the data? 